# Processamento de Imagens — Atividade 2

**Objetivo:** analisar três fotografias com exposições diferentes por meio dos histogramas em tons de cinza e aplicar uma operação de correção adequada a cada caso.

Foram usadas três fotos próprias: uma cena **escura/subexposta**, uma cena com **superexposição localizada** próxima à janela e uma imagem **bem exposta**.

A operação escolhida foi **correção de gamma**, usando a transformação $s = 255(r/255)^\gamma$. Nesta convenção, $\gamma < 1$ clareia a imagem e $\gamma > 1$ escurece os tons médios e altos.

In [1]:
import base64
import cv2
import numpy as np
import matplotlib.pyplot as plt

IMAGENS_B64 = {
    "escura": "/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAA4KCw0LCQ4NDA0QDw4RFiQXFhQUFiwgIRokNC43NjMuMjI6QVNGOj1OPjIySGJJTlZYXV5dOEVmbWVabFNbXVn/2wBDAQ8QEBYTFioXFypZOzI7WVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVn/wAARCACAAGADASIAAhEBAxEB/8QAGgAAAwEBAQEAAAAAAAAAAAAAAAECAwQFB//EACwQAAIBAwIFAgUFAAAAAAAAAAABAgMRIRIxBCJBUWEFMhNCkbHBM1JxgfD/xAAaAQADAQEBAQAAAAAAAAAAAAAAAQIDBAUG/8QAJBEAAgICAgEEAwEAAAAAAAAAAAECEQMhBBJBBRMxUSJhcfD/2gAMAwEAAhEDEQA/APnohgQdQgAAEAAAAAmMTGJiGIAEAAACLEMQjQAAAEIAABDAAAYmIoQxUIChAKhgACKAAAAEAblACQgGAFUKwDABUIBgAUIBDAQAAwAujTVSelyUe3linF05uEt07FOlONk2oybwm7XKqR0w0zm3UX+yTezfp+O1TX+oyOnh+CqVuaXJDu92b+ncLqtVqK6XtT+56UrOL6eTOc60j0eJwO8fcyfH0c9Pg+Hppcib85HOhSm7OnF/0baW9yVhyXzfgxtnr+1jS69VX8OSr6dTcXovGXTODzJwlTm4yVmj3G7Hm+pfqwfg0xybdM8rn8fHGHeCqjhGIZ0HiocYuUkluyr6VaLV83ZvQlGhSc2k5Sxv7UcqeqWlLfbuStmskoJb2xzlKTim76VZeC8t92yIKzd91g0i9MlLs7jYob2z3KNPRThFfLGzKkrNN7XFCSlTT6PIalqUfqch9cuqikhvci+q7e3Qt6exlteL2WwCloHp7Hm+o+6HfJ21JqEW29jya9V1KjbNMS3Z5PqGVKHTyyAEM6DxEU5cijFXbecEX58YsXB6Zp3tZlSaUviRTst2+rAqrV2RHdlpNuyV2yUt2+uQUruyeldX2FQ06R2UuJlS0wb1xT6fg7uHnGa+IpJ6t/B4mp67QwnhfwbRbh7W0Zzh9HfxuZKD3tI9qTSzdHJX4mMNss4XVn+5mTzkmOL7N83qDaqCoutXlUfg52UyGbpUeNkm5O2yhgAAiorVJRXXBpWjKGik0ko5x1Zjd9BNt8qFWy+yUWvI23J2j9RtfL2ywdoxst/uJXfL1bvJlEfpl01hy77FiWwyGdEVSoTJZTIY0TIlkMpkso5pFDEJvotwKuht3whq0UCVhPmduiAPjfkXl7s1hGyzuyILVK/RGopMvHHyAABBsJkspkstGcmQyGWyWXRzSZ//2Q==",
    "clara": "/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAA4KCw0LCQ4NDA0QDw4RFiQXFhQUFiwgIRokNC43NjMuMjI6QVNGOj1OPjIySGJJTlZYXV5dOEVmbWVabFNbXVn/2wBDAQ8QEBYTFioXFypZOzI7WVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVn/wAARCACAAIADASIAAhEBAxEB/8QAGwAAAwEBAQEBAAAAAAAAAAAAAwQFAgEGAAf/xAA0EAABBAEDAwMBBQcFAAAAAAABAAIDEQQSITEFQVETImFxBhQygaFCUmKRscHRIyQzcvH/xAAZAQADAQEBAAAAAAAAAAAAAAABAgMEAAX/xAAeEQEBAQEAAgMBAQAAAAAAAAAAAQIRITEDEkETUf/aAAwDAQACEQMRAD8AemZQOk2DvSnZbu3YKnlPAcQ2gpGQQ53CzbnlsxfBR9XZpBkGk62j4I8or+RX8vCyX0dN2uzHaYI4I4PC2xuokEbrkbbeQ7hOMxi4aRu74WiYQumWsot08FPY5a9xjBBe38Tb3C5j4z3ENr3N3+qT+0OPJgyw5uO4skbsSO4SX4+mnyc8r0bKaB4WMhoN/IopXonWoOoARTlsWT+72d9P8I3W8n7jj6tOp7va0fKxXGpvjXN5uOvIdIyDidVxpx+xIL+l7r9JzSHO1DSWgc2vzCBhE8Qdtbt/5r0+Xny9WyntiJixIzpLxzIvVjyrO01MC2Imq1HYfpyvSQs0QRs7taB+ihyY8UeLgvY52hzqeHG9wqbM9h7qOlczsOFm6A4Gt2gHuLXfvrDvYQpJw7hA3HnsuUajR+inPk9xv/1MZRJkcflJydxtaz29apOMO7DZZqzZWXXZor5pPe68hPguh4r1eSquCwOo9geFJiGo0HbqxhMcHjYlbcMXyVcxcYOLSF4n7QZz8vqeREH1Ax2ho5G3cKh9o+vOaw9Ow3OYQamdwf8AqF5YNJXb53wXHeeRI3MiNhuojuU07Lny3ASyOeG8BzrSWg+EaFwaaI/NL9Z7NdXnDHLTY4CdxckRSMiAsNrUB38pT1W6dqPwULcW/uSm4V6fq2fCYMduOAWhzjQ/Z8JBmd8kJPEldpO2odwiRyC3D0muF9yUlx2rZ+T6zh4Z4IABKOzLdXKUgjMo0sxmvP8ACD/VNDCjjGrJIj/gjOp3+Ah/Mf6vuoQ6NThx3UiRwskqv1KQBxYOOVIlA7jlY2r8CPO3Cw+Sl14pBk4T5Lp372Y3aq4VLp/2ijgePVjcR8KBMdkJota8Wxk+SSmMzIOZnT5DuZHlywAuiMtHkd12qRTHxzqPuAI+iO9kfgNvuFyFmmML6TdwHhFznpb0d/kLp0gVe65vqobIrKcakbf8XdNAF6f7Mhrh2Krvw2R5OrSHMPAPCTwscFwLXBwXqI8USSBj/wBwceQuD0pYXR4zgxa3FriLOk7JbI+zRebjnB+HBegh0+izQKbpFD4WlHtUfmvUx/rHspsg7qz1KnWf1USTmlkb56Acd0CQ2jvI4CVeRfKpmE1S053XWRf7cyk1uAFmbd/lNZrDDj40R5LS8q8ZdOY8oBp4sLb2MZIGglzHbixuEBw8dqCLBudb962A8pyHBWj6crAFusr5wqMeb3WAwlx91AUjAbr3FEZs21hzHNdbXNczsbW45C13kfTlNAO4THOcNLiCvV9MjyWkH0wewPIP9wovR4o8iRrYjT/3fP0XuMZjMURQgXK4b/AQ1rjudHxtYhaJGaHDartFK6s76vilGqPzudwcx138BR5h7k9PLY2tIPNWSszbC8poJcncosptBVMxPVAcLmbQ7pzrAJmhd20UhYkfqZjb3rdUOpQmbH2/EzcKneWI2dlS2iw4+aX0LjGfgr6AGRrgOaXQdOzgdlVLgpnJPuF0s6nPcTZq7pfNYXybAmzsmnwiFhLu4R6EgALr7o8WpxAa2yhMIDDe5OwH91a6FjCXKiaR+I7o945W6J0sYrfvmWDbW6w0bfT+a9Jhuc55mLtUjzbj2HwPhQ83JOXkux4tomPGs9tuAquMBiQCSR+x4HlR3rzw2Z46tNmjc4tDhqHIPK03uT3K8ZPmPGeZQSNXa1cwOqh4DJBY8pe+TceCkcLJNJSVws2iPdtWyVldyoSNdoMjghFwAWnHdBeeytIlqqHSo9T3PvbhVXtBaVP6ex0LGm7B5HhUpfw2EtvkJ6T4cNrJ3SA0Hfs+ESbB1gkAFEDuCjxyA+2+UftQ+sS3MdijUdI7C0o6UvdqebVfK6cMrI1CRwofktQ9Bjd/yTuo9gFSbzPadzUZlGQAmh5+F6PpDcgyB+O0sYBXqOH9Amcbo+Fj6XemXkd3bqvH+EUBQS6+X/HTH+tY2HGyIAbDye5+VnMm9RrY27NZwVvJeWNocFLkjRZ5ULryrMp2YTrbZquyPiyEUlct2qyfOy+x3GwLpV/A/UJ5r8ktI6/hH6gTj5EkbjwdlNMjpHUDSXOVNajTn702yfCLDjO1B7xv2C3FHHEATu/yiuk8J+pe3WSviIs2O4TLpiYtbEq7do2TTWaYTfhLRYimLm1tyjFrngNH4r7dkhA8N1O25TsU4L2+6jaNjpVTEjcAGyG1SjjjG4FpOAhzdymGmhtwpUxsAabCKwWA4cgUkw8xg7amnwuGYgExuseO4SuMTOLpNJ4S80lMNcLpmLgX1eym5uR6bSLoo5naN9A5Em2juUbGoDy7v8KdGXSusp6L2NoCqVr4nCQv9ocESujyGg7inUofomPhq9kWjIx3MPjZeYmPvLCKIQldYU0kC7R4ItX5oT5G69LdynMaRsbd+V1CDOgbGGl3CDn5AZBQO6Fl5ZfsNgEo6UE2aK7M/a61iBr3t22Hkp6JgYQ5zgSEBkpPa0eLfb9U1oSHYuoSRcBpCfxeqF5AcwH6FSvTB4HK60GKiD33U7JVIvDLjLqBIP0Q3TF7qFEjv8KeJC/kWUQvMQ9TUl443LOIYvca+qizZDsqbb8I2S2TlvyZdN+1EhprSq5zwlvT+O3SzZMB1jYpRj/aQD2RWXQ8Lhf/2Q==",
    "bem": "/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAA4KCw0LCQ4NDA0QDw4RFiQXFhQUFiwgIRokNC43NjMuMjI6QVNGOj1OPjIySGJJTlZYXV5dOEVmbWVabFNbXVn/2wBDAQ8QEBYTFioXFypZOzI7WVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVlZWVn/wAARCACAAGADASIAAhEBAxEB/8QAGgAAAgMBAQAAAAAAAAAAAAAAAQIDBAUGAP/EACsQAAICAgEDAwMEAwEAAAAAAAECAAMEESEFEjEGE0EiYXEUUYGxIzLxYv/EABkBAQEBAQEBAAAAAAAAAAAAAAABAgMEBf/EACQRAAICAgEDBAMAAAAAAAAAAAABAgMRIRIxYcEEIkFxEyMy/9oADAMBAAIRAxEAPwC4slWRLJVnM6kgjiII4gDiMIojCAMBDqeEMgAYpjGKYADFMYxTAM9ZMsgQyVZQSiOIixxAHEYRRGEAaGLGgHjFhggAMUwwGAZySZZEkmWCEixxEUQPlU1EhiSR50PEqTk8IxOyMFmTwTiM7pVW1ljBUUbLHwJ6tlsrFiMChG9zM9TgN0Gwh9AsuiOQeY+cG86yifF6xg5d4qqtPefHcpG/xL5GpxNGAt2T0+rDuF+QxD22J/rWAZ3pqDcgxLQTyVzFMldNSMiQohikxjFMApIJMokaCTKIIOnmZGYpx7shXsBr37gA5Oz8TZQSlm2UV5ypbQpNqcWMeN+JVOUNx8eTzeqqjZDfnwUbryvQAq2Iotv7CzHQ15P9TDzs61sNsT3nalG4HgEfH8fM6HqNKXFMa6xMjXCgDWif6/MoX9MVRViWV2ipOfpbZ2B95qM4We6DznZiuz8f6pprGt/JmdC6nk9PuCVuntO4LgqOR+fIna4nWMTMzLMfGtLOg7t64YfaZeB0/pyWpkKSLKq9aPCnQ13EfvqZvSKXw+p5GYiAU/UtY8KQTOFdjsslDi1g9ErYQgpuSwzsWY75iGVcXqVN9ntWEV2jj/yT9jLbDU6dDcZqazFiGIYximDRXUSVRAokgEEGUTL9Sh16fXalZc1v9RA/1B+f6mssZhtCp8EajOw1lYZwVOfjXELpqbgfpZOOf3mljdTy6bLLb6VylVNd5+PvKHqTpeJ01arKGfvc67Cd+By24/R68jK6dYdHt2VU7+06uNdqxNHz7K51e6vfbqS1v+oyzYCPaQeD4J/EstYlliKS91h4VRwP4EpYubVl5Yxk4ZwduBxwJ5uiZRt7hlAaJKk74E7uyMFhHlr9JOx5lpF3Lttoauu2tUI/ydpO9/maNXUMmulFFXu2OdrWASQvxzI+j4pxqbBmMt91vDMxLfT8DmDJpau411C2wOoWkhuE55Bnjl/SaX33PeqnUnxf12NWi431bdVSxTpkDA6jGV8DCTERiX77W8keP4llpZYzo9NfLiufUUCOBPKI4Eho8BH1sQAR1HMgOH9XK1/V6aKz3v7YHaPgk/8AJsYdIw8WrHUD6Bz9z8zNsVc31PkWn6kqfj7kcCdHThsV7m8mafQiOGxaWxvUS/GriNfY/wDZ1bNzMfqyvX6npVAAV7RvXPPmdGuL3LvUMIqqCfBh/wAifOxLaY2jJ/YBEhSnTaxOiJbB2IBQAdx+3XiCjCMIBGEhAiV+o5qdPwbMhuSo0o/dj4EsRL6Ksio13ItiHyrCUHP+kMZnpyMu4A+4/wBO/wB/kzqBIaa0prWupAiKNBVHAkinmHsI5XMX9X6yNdZBKKB+CBOpVAiBf2nM9GwsxPUmRk5NLKPrPefBJPGp05hhC6EBEMBkKKYhkhiGAMIYNRhBD0PxBGAgHgI2oNQwU9BDPGALAYTAYAsUxoCIB//Z"
}

def carregar_cinza(nome):
    dados = base64.b64decode(IMAGENS_B64[nome])
    arr = np.frombuffer(dados, dtype=np.uint8)
    img_bgr = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

def corrigir_gamma(img, gamma):
    tabela = np.array([255 * ((i / 255.0) ** gamma) for i in range(256)])
    tabela = tabela.clip(0, 255).astype(np.uint8)
    return cv2.LUT(img, tabela)

def histograma(img):
    return cv2.calcHist([img], [0], None, [256], [0, 256]).ravel()

def figura_antes_depois(img, corrigida, titulo, gamma):
    fig, ax = plt.subplots(2, 2, figsize=(10, 7))
    ax[0, 0].imshow(img, cmap='gray', vmin=0, vmax=255)
    ax[0, 0].set_title('Antes — tons de cinza')
    ax[0, 0].axis('off')
    ax[0, 1].imshow(corrigida, cmap='gray', vmin=0, vmax=255)
    ax[0, 1].set_title(f'Depois — correção gamma (γ={gamma})')
    ax[0, 1].axis('off')
    ax[1, 0].plot(np.arange(256), histograma(img))
    ax[1, 0].set_xlim(0, 255)
    ax[1, 0].set_title('Histograma — antes')
    ax[1, 0].set_xlabel('Intensidade')
    ax[1, 0].set_ylabel('Número de pixels')
    ax[1, 1].plot(np.arange(256), histograma(corrigida))
    ax[1, 1].set_xlim(0, 255)
    ax[1, 1].set_title('Histograma — depois')
    ax[1, 1].set_xlabel('Intensidade')
    ax[1, 1].set_ylabel('Número de pixels')
    fig.suptitle(titulo)
    plt.tight_layout()
    plt.show()

def diferenca_media(a, b):
    return float(np.mean(np.abs(a.astype(np.float32) - b.astype(np.float32))))

## 1. Imagem escura

**Diagnóstico:** o histograma está fortemente concentrado nas baixas intensidades: aproximadamente **91,7% dos pixels estão entre 0 e 63**, e o pico fica próximo da intensidade **11**. Isso indica subexposição e pouca separação tonal nas regiões de sombra.

**Operação escolhida:** correção de gamma com **γ = 0,55**. Como γ é menor que 1, os tons escuros são deslocados para intensidades maiores, clareando a imagem sem aplicar uma expansão linear agressiva em toda a faixa.

In [2]:
img_escura = carregar_cinza('escura')
gamma_escura = 0.55
escura_corrigida = corrigir_gamma(img_escura, gamma_escura)
figura_antes_depois(img_escura, escura_corrigida, 'Imagem escura', gamma_escura)

## 2. Imagem clara / superexposta

**Diagnóstico:** o histograma ocupa uma faixa ampla, mas apresenta uma parcela de pixels nas intensidades altas e pixels saturados em **255** (aproximadamente **0,7%** nesta versão analisada). Isso corresponde ao estouro de altas luzes próximo à janela; os detalhes já recortados em 255 não podem ser recuperados por uma transformação tonal.

**Operação escolhida:** correção de gamma com **γ = 1,20**. Como γ é maior que 1, a transformação reduz principalmente as intensidades médias e altas, diminuindo a sensação de excesso de luminosidade sem tentar inventar informação nas regiões já saturadas.

In [3]:
img_clara = carregar_cinza('clara')
gamma_clara = 1.20
clara_corrigida = corrigir_gamma(img_clara, gamma_clara)
figura_antes_depois(img_clara, clara_corrigida, 'Imagem clara / superexposta', gamma_clara)

## 3. Imagem bem exposta

**Diagnóstico:** o histograma se concentra principalmente nos tons médios: aproximadamente **92,2% dos pixels estão entre 64 e 191**, com pico próximo da intensidade **182**. Não há concentração relevante nos extremos 0 ou 255, indicando que a maior parte dos detalhes está preservada.

**Operação escolhida:** correção de gamma com **γ = 1,05**. Como a imagem já apresenta exposição adequada, foi usado um ajuste muito suave, próximo de γ = 1, para evitar modificar desnecessariamente uma distribuição tonal que já é satisfatória.

In [4]:
img_bem = carregar_cinza('bem')
gamma_bem = 1.05
bem_corrigida = corrigir_gamma(img_bem, gamma_bem)
figura_antes_depois(img_bem, bem_corrigida, 'Imagem bem exposta', gamma_bem)

## Comparação final

In [5]:
diferencas = {
    'Escura (γ=0,55)': diferenca_media(img_escura, escura_corrigida),
    'Clara / superexposta (γ=1,20)': diferenca_media(img_clara, clara_corrigida),
    'Bem exposta (γ=1,05)': diferenca_media(img_bem, bem_corrigida),
}

for nome, valor in diferencas.items():
    print(f'{nome}: diferença média absoluta = {valor:.2f} níveis de cinza')

Escura (γ=0,55): diferença média absoluta = 37.04 níveis de cinza
Clara / superexposta (γ=1,20): diferença média absoluta = 15.15 níveis de cinza
Bem exposta (γ=1,05): diferença média absoluta = 3.90 níveis de cinza


**Resposta:** a operação fez menos diferença na **imagem bem exposta**. O histograma original já estava majoritariamente distribuído em tons médios, sem concentração relevante nos extremos 0 e 255, portanto não havia necessidade de uma correção forte. Por isso foi usado γ = 1,05, muito próximo de 1, produzindo apenas um ajuste discreto; nas imagens escura e superexposta foram necessários valores de gamma mais afastados de 1 para deslocar de forma perceptível as intensidades.